In [1]:
#Installed the necessary transformers and packages
!pip install transformers torch torchvision --quiet
!pip install ultralytics --quiet
!pip install gradio --quiet
!pip install gtts --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.7 MB/s eta 0:00:00


In [2]:
#import transformers and packages and gTTs (google Text-Speech )
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from ultralytics import YOLO
import gradio as gr
import torch
from gtts import gTTS
import tempfile


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Setting up the BlipProcessor for image captioning with error handling
try:
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model_caption = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
    print("✓ BLIP model loaded successfully")
except Exception as e:
    print(f"✗ Error loading BLIP model: {str(e)}")
    print("Please check your internet connection and try again.")
    raise


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [4]:
# Setting up the YOLO Model for object detection with error handling
try:
    model_yolo = YOLO("yolov8n.pt")  # Use the lightweight version
    print("✓ YOLO model loaded successfully")
except Exception as e:
    print(f"✗ Error loading YOLO model: {str(e)}")
    print("Please check your internet connection and try again.")
    raise


In [5]:
# Danger keyword categories for threat detection
# Crime scene indicators: severe threats requiring immediate attention
CRIME_SCENE_KEYWORDS = [
    'knife', 'blood', 'fight', 'handcuffs', 'gun', 'weapon', 'crime scene', 'crime'
]

# Police-related keywords: may indicate law enforcement presence (caution if alone)
POLICE_KEYWORDS = ['police', 'police car', 'police light']

# General danger keywords: potential hazards requiring caution
GENERAL_DANGER_KEYWORDS = [
    'fire', 'explosion', 'smoke', 'car', 'truck', 'bus', 'train',
    'aggressive crowd', 'bear', 'dog being aggressive', 'cliff', 'motorcycle',
    'rain', 'drugs', 'danger sign', 'broken', 'glass', 'ambulance'
]

# Combined list for general matching (backward compatibility)
# All keywords are stored in lowercase for case-insensitive matching
danger_keywords = CRIME_SCENE_KEYWORDS + POLICE_KEYWORDS + GENERAL_DANGER_KEYWORDS


In [6]:
# Take an image and generate a text description
def generate_caption(image):
    """Generate a text caption for an image using BLIP model.
    
    Args:
        image: PIL Image object
    
    Returns:
        str: Generated caption or error message
    """
    try:
        # Validate input
        if image is None:
            return "Error: No image provided"
        
        # Process and generate caption
        inputs = processor(images=image, return_tensors="pt")
        out = model_caption.generate(**inputs, no_repeat_ngram_size=2)
        return processor.decode(out[0], skip_special_tokens=True)
    except Exception as e:
        return f"Error generating caption: {str(e)}"


In [7]:
import os

def text_to_audio(text_input):
    """Convert text to audio using Google Text-to-Speech.
    
    Args:
        text_input: Text string to convert to audio
    
    Returns:
        str: Path to generated audio file or None on error
    
    Note:
        Temporary audio files are created and should be cleaned up by Gradio.
        If using outside Gradio, implement manual cleanup.
    """
    try:
        # Validate input
        if not text_input or not text_input.strip():
            print("Warning: Empty text input for audio generation")
            return None
        
        # Generate audio
        tts = gTTS(text=text_input, lang='en')
        
        # Create temporary file
        # Note: delete=False needed for Gradio to access the file
        # Gradio handles cleanup of temporary files automatically
        temp_audio_file = tempfile.NamedTemporaryFile(delete=False, suffix='.mp3')
        temp_audio_file_path = temp_audio_file.name
        temp_audio_file.close()

        # Save audio to file
        tts.save(temp_audio_file_path)
        return temp_audio_file_path
    except Exception as e:
        print(f"Error generating audio: {str(e)}")
        return None

print("text_to_audio function defined successfully.")

text_to_audio function defined successfully.


In [8]:
def detect_danger(image):
    """Detect dangerous objects or situations in an image using YOLO.
    
    Args:
        image: PIL Image object
    
    Returns:
        str: Safety status message with detected objects
    
    Detection precedence:
        1. Crime scene indicators (knife, blood, gun, etc.) -> 🔴 Crime Scene
        2. Police presence alone (without crime indicators) -> 🟡 Caution
        3. General dangers (fire, vehicles, etc.) -> 🟡 Caution
        4. No threats detected -> 🟢 Safe
    """
    try:
        # Validate input
        if image is None:
            return "Error: No image provided for danger detection"
        
        # Run YOLO detection
        results = model_yolo(image)
        labels = []
        for r in results:
            labels += [model_yolo.names[int(c)] for c in r.boxes.cls]

        # Match detected objects against all danger keywords
        matched = [item for item in labels if item.lower() in danger_keywords]

        # Priority 1: Check for crime scene keywords (highest severity)
        crime_detected = [word for word in matched if word.lower() in CRIME_SCENE_KEYWORDS]
        if crime_detected:
            return f"🔴 Possible Crime Scene or danger: {', '.join(set(matched))}"
        
        # Priority 2: Check for police presence alone (caution level)
        police_detected = [word for word in matched if word.lower() in POLICE_KEYWORDS]
        if police_detected:
            return f"🟡 Caution: {', '.join(set(matched))}"
        
        # Priority 3: Check for other general danger keywords
        general_danger_detected = [word for word in matched if word.lower() in GENERAL_DANGER_KEYWORDS]
        if general_danger_detected:
            return f"🟡 Caution: {', '.join(set(matched))}"
        
        # No threats detected
        return "🟢 Safe"
    except Exception as e:
        return f"Error detecting dangers: {str(e)}"

In [9]:
def analyze_image(img):
    """Main function to analyze an image and provide caption, safety status, and audio.
    
    Args:
        img: PIL Image object
    
    Returns:
        tuple: (text_output, audio_path) containing results and audio file path
    """
    try:
        # Validate input
        if img is None:
            error_msg = "Error: No image provided. Please upload an image."
            return error_msg, None
        
        # Generate caption
        caption = generate_caption(img)
        
        # Check if caption generation failed
        if caption.startswith("Error"):
            return f"Caption: {caption}\n\nSafety Status: Unable to analyze", None
        
        # Detect dangers
        danger = detect_danger(img)
        
        # Check if danger detection failed
        if danger.startswith("Error"):
            return f"Caption: {caption}\n\nSafety Status: {danger}", None

        # Optional: flag certain keywords from the caption
        crime_words = ['arrest', 'weapon', 'blood', 'shooting', 'gun', 'knife']
        if any(word in caption.lower() for word in crime_words):
            danger = "🔴 Possible Crime Scene or danger (based on caption)"

        # Extract spoken danger status for audio narration
        spoken_danger_status = "Unknown Safety Status"
        if danger.startswith("🔴 Possible Crime Scene"):
            spoken_danger_status = "Possible Crime Scene"
        elif danger.startswith("🟡 Caution"):
            spoken_danger_status = "Caution"
        elif danger == "🟢 Safe":
            spoken_danger_status = "Safe"

        # Combine text for narration
        combined_text = f"The image shows: {caption}. Safety status: {spoken_danger_status}"
        audio_path = text_to_audio(combined_text)

        return f"Caption: {caption}\n\nSafety Status: {danger}", audio_path
    except Exception as e:
        error_msg = f"Error analyzing image: {str(e)}"
        print(error_msg)  # Log for debugging
        return error_msg, None

In [10]:
#Utlizing Gradio library to create a user friendly web interface
import gradio as gr
gr.Interface(
    fn=analyze_image,
    inputs=gr.Image(type="pil"),
    outputs=[gr.Textbox(label="Caption Results: ", lines=10),gr.Audio(type="filepath", label="Audio Narration")],
    title=" Seeing Through Words App",
    description="Upload an image to receive a text/audio description and a danger warning"
).launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b0364b3b5841a74cea.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
